# AI Portfolio Analyzer — Colab Training Notebook

Train the LSTM / Transformer model on Google Colab (free GPU).

> Educational use only — not financial advice.

## Cell 1 — Check GPU

Go to **Runtime → Change runtime type → T4 GPU** before running.

In [1]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU detected. Go to Runtime -> Change runtime type -> GPU (T4 recommended)')

try:
    import torch
    print(f'PyTorch version : {torch.__version__}')
    print(f'CUDA available  : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU             : {torch.cuda.get_device_name(0)}')
        print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('PyTorch not yet installed — will be installed in Cell 5.')


Sun May 17 17:44:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Mount Google Drive

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ai-portfolio-analyzer'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/ai-portfolio-analyzer


## Cell 3 — Upload project files

Upload `ai-portfolio-analyzer.zip` to your Google Drive root, then run this cell.
Skip if already unzipped in Drive.

In [4]:
import os

ZIP_PATH = '/content/drive/MyDrive/ai-portfolio-analyzer.zip'

if os.path.exists(ZIP_PATH):
    os.system(f'unzip -q -o "{ZIP_PATH}" -d /content/drive/MyDrive/')
    print('Unzipped project to Drive.')
elif os.path.isdir(PROJECT_DIR) and os.listdir(PROJECT_DIR):
    print('Project folder already exists in Drive — skipping unzip.')
else:
    print('Zip not found at:', ZIP_PATH)
    print('Upload ai-portfolio-analyzer.zip to your Drive root, then re-run.')


Unzipped project to Drive.


## Cell 4 — Copy project to /content (fast I/O)

In [5]:
import shutil, sys, os

CONTENT_DIR = '/content/ai-portfolio-analyzer'

if not os.path.exists(CONTENT_DIR):
    shutil.copytree(PROJECT_DIR, CONTENT_DIR)
    print(f'Copied project -> {CONTENT_DIR}')
else:
    print(f'Project already at {CONTENT_DIR} (skipping copy)')

os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)
print('Working dir:', os.getcwd())


Project already at /content/ai-portfolio-analyzer (skipping copy)
Working dir: /content/ai-portfolio-analyzer


## Cell 5 — Install dependencies

Takes ~3-5 min on first run. After this cell finishes, do **Runtime → Restart session**, then continue from Cell 6.

In [6]:
import subprocess

# Pin numpy to avoid conflicts with Colab pre-installed packages
subprocess.run(['pip', 'install', '-q', 'numpy>=2.1', '--upgrade', '--force-reinstall'], check=True)

subprocess.run([
    'pip', 'install', '-q',
    'pandas>=2.2', 'scikit-learn>=1.5', 'scipy>=1.14',
    'yfinance>=0.2.40',
    'torch>=2.1.0',
    'xgboost>=2.0', 'lightgbm>=4.0',
    'transformers>=4.38.0', 'accelerate>=0.27.0', 'sentencepiece>=0.1.99',
    'cvxpy>=1.4',
    'plotly>=5.18.0', 'streamlit>=1.32.0',
    'wandb>=0.16.0',
    'pyyaml>=6.0', 'python-dotenv>=1.0',
    'requests>=2.31.0', 'feedparser>=6.0.10',
    'pyarrow>=14.0', 'fastparquet',
], check=True)

print('All packages installed. Now do Runtime -> Restart session, then continue from Cell 6.')


All packages installed. Now do Runtime -> Restart session, then continue from Cell 6.


## Cell 6 — Apply source code fixes

Fixes `torch.load` compatibility and W&B `reinit` argument for newer package versions.

In [19]:
import os, shutil, sys

CONTENT_DIR = '/content/ai-portfolio-analyzer'
os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

# --- Fix 1: torch.load weights_only in train.py ---
with open('src/train.py', 'r') as f:
    content = f.read()
content = content.replace(
    'ckpt = torch.load(best_ckpt, map_location=device)',
    'ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)'
)
with open('src/train.py', 'w') as f:
    f.write(content)
print('Fix 1 applied: train.py torch.load')

# --- Fix 2: W&B reinit argument in utils.py ---
with open('src/utils.py', 'r') as f:
    content = f.read()
# Handle both possible variants (underscore or space)
content = content.replace('reinit="finish_previous",', 'reinit="finish_previous",')
content = content.replace('reinit="finish previous",', 'reinit="finish_previous",')
with open('src/utils.py', 'w') as f:
    f.write(content)
print('Fix 2 applied: utils.py wandb reinit')

# --- Verify fixes ---
import subprocess
r1 = subprocess.run(['grep', '-n', 'torch.load', 'src/train.py'], capture_output=True, text=True)
r2 = subprocess.run(['grep', '-n', 'reinit', 'src/utils.py'], capture_output=True, text=True)
print('train.py torch.load line :', r1.stdout.strip())
print('utils.py reinit line     :', r2.stdout.strip())


Fix 1 applied: train.py torch.load
Fix 2 applied: utils.py wandb reinit
train.py torch.load line : 247:    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
utils.py reinit line     : 164:            reinit="finish_previous",


## Cell 7 — W&B login

In [18]:
USE_WANDB = True  # set False to skip W&B

if USE_WANDB:
    import wandb
    wandb.login()  # paste your API key from wandb.ai/authorize
    api = wandb.Api()
    WANDB_ENTITY = api.viewer.entity
    print(f'Logged in as: {WANDB_ENTITY}')
else:
    import os
    os.environ['WANDB_DISABLED'] = 'true'
    WANDB_ENTITY = ''
    print('Skipping W&B.')


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Logged in as: vedantdaga-


## Cell 8 — Load config & configure for Colab

In [20]:
import os, sys
CONTENT_DIR = '/content/ai-portfolio-analyzer'
os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)
print('Working dir:', os.getcwd())

import os, importlib
import src.utils
src.utils = importlib.reload(src.utils)
from src.utils import load_config, set_seed, detect_device

cfg = load_config('configs/config.yaml')

# --- Model ---
cfg['model']['type']            = 'lstm'
cfg['model']['hidden_size']     = 128
cfg['model']['num_layers']      = 2
cfg['model']['dropout']         = 0.3
cfg['model']['use_uncertainty'] = True

# --- Training ---
cfg['training']['epochs']                  = 60
cfg['training']['batch_size']              = 128
cfg['training']['learning_rate']           = 0.001
cfg['training']['early_stopping_patience'] = 15
cfg['training']['checkpoint_dir']          = 'checkpoints'

# --- Data paths ---
cfg['data']['raw_dir']       = 'data/raw'
cfg['data']['processed_dir'] = 'data/processed'

# --- Disable live sentiment (no API key needed for training) ---
cfg['features']['use_sentiment'] = False

# --- W&B (entity auto-detected from login) ---
cfg['wandb']['enabled'] = USE_WANDB
cfg['wandb']['entity']  = WANDB_ENTITY

# --- Ensure dirs exist ---
for d in ['data/raw', 'data/processed', 'checkpoints', 'reports/figures']:
    os.makedirs(d, exist_ok=True)

device = detect_device()
set_seed(cfg['training']['seed'])

print(f"Device : {device}")
print(f"Model  : {cfg['model']['type']}  hidden={cfg['model']['hidden_size']}  layers={cfg['model']['num_layers']}")
print(f"Epochs : {cfg['training']['epochs']}  Batch: {cfg['training']['batch_size']}  LR: {cfg['training']['learning_rate']}")
print(f"W&B    : {cfg['wandb']['enabled']}  entity={cfg['wandb']['entity']}")


Working dir: /content/ai-portfolio-analyzer
Device : cuda
Model  : lstm  hidden=128  layers=2
Epochs : 60  Batch: 128  LR: 0.001
W&B    : True  entity=vedantdaga-


## Cell 9 — Download raw market data

In [21]:
import importlib, src.dataset
src.dataset = importlib.reload(src.dataset)
from src.dataset import download_all

raw_data = download_all(cfg)

print(f'\n{len(raw_data)} tickers loaded:')
for ticker, df in sorted(raw_data.items()):
    print(f'  {ticker:6s}  {len(df):5,d} rows  '
          f'{df.index[0].date()} -> {df.index[-1].date()}')



46 tickers loaded:
  AAPL    3,773 rows  2010-01-04 -> 2024-12-30
  AMD     3,773 rows  2010-01-04 -> 2024-12-30
  AMZN    3,773 rows  2010-01-04 -> 2024-12-30
  BAC     3,773 rows  2010-01-04 -> 2024-12-30
  COST    3,773 rows  2010-01-04 -> 2024-12-30
  CRM     3,773 rows  2010-01-04 -> 2024-12-30
  CVX     3,773 rows  2010-01-04 -> 2024-12-30
  DIA     3,773 rows  2010-01-04 -> 2024-12-30
  DIS     3,773 rows  2010-01-04 -> 2024-12-30
  EWJ     3,773 rows  2010-01-04 -> 2024-12-30
  EWZ     3,773 rows  2010-01-04 -> 2024-12-30
  FXI     3,773 rows  2010-01-04 -> 2024-12-30
  GLD     3,773 rows  2010-01-04 -> 2024-12-30
  GOOGL   3,773 rows  2010-01-04 -> 2024-12-30
  GS      3,773 rows  2010-01-04 -> 2024-12-30
  HD      3,773 rows  2010-01-04 -> 2024-12-30
  HYG     3,773 rows  2010-01-04 -> 2024-12-30
  INTC    3,773 rows  2010-01-04 -> 2024-12-30
  IWM     3,773 rows  2010-01-04 -> 2024-12-30
  JNJ     3,773 rows  2010-01-04 -> 2024-12-30
  JPM     3,773 rows  2010-01-04 -> 2024

## Cell 10 — Build processed feature datasets

Set `TRAIN_TICKER = 'ALL'` to train on all 46 tickers.

In [22]:
from src.dataset import build_processed_dataset

TRAIN_TICKER = 'ALL'  # change to any ticker, or 'ALL'

if TRAIN_TICKER == 'ALL':
    for t in cfg['data']['tickers']:
        try:
            df_p = build_processed_dataset(t, cfg, sentiment_df=None)
            print(f'  {t:6s}  {df_p.shape}')
        except Exception as e:
            print(f'  {t:6s}  ERROR: {e}')
    print('\nAll tickers processed.')
else:
    df_proc = build_processed_dataset(TRAIN_TICKER, cfg, sentiment_df=None)
    print(f'Processed dataset shape: {df_proc.shape}')
    first_cols = list(df_proc.columns[:8])
    extra = len(df_proc.columns) - 8
    print(f'Columns: {first_cols} ... (+{extra} more)')
    print(df_proc.describe().T[['mean', 'std', 'min', 'max']].round(4))


  AAPL    (3522, 52)
  MSFT    (3522, 52)
  NVDA    (3522, 52)
  TSLA    (3400, 52)
  AMZN    (3522, 52)
  META    (2923, 52)
  GOOGL   (3522, 52)
  AMD     (3521, 52)
  INTC    (3522, 52)
  QCOM    (3522, 52)
  MU      (3522, 52)
  CRM     (3522, 52)
  ORCL    (3522, 52)
  NFLX    (3522, 52)
  PYPL    (2138, 52)
  JPM     (3522, 52)
  BAC     (3522, 52)
  GS      (3522, 52)
  V       (3522, 52)
  MA      (3522, 52)
  UNH     (3522, 52)
  JNJ     (3522, 52)
  WMT     (3522, 52)
  COST    (3522, 52)
  HD      (3522, 52)
  DIS     (3522, 52)
  XOM     (3522, 52)
  CVX     (3522, 52)
  SPY     (3522, 52)
  QQQ     (3522, 52)
  IWM     (3522, 52)
  DIA     (3522, 52)
  XLF     (3522, 52)
  XLK     (3522, 52)
  XLE     (3522, 52)
  XLV     (3522, 52)
  XLI     (3522, 52)
  EWJ     (3522, 52)
  EWZ     (3522, 52)
  FXI     (3522, 52)
  TLT     (3522, 52)
  HYG     (3522, 52)
  LQD     (3522, 52)
  GLD     (3522, 52)
  SLV     (3522, 52)
  UVXY    (2754, 52)

All tickers processed.


## Cell 11 — Train the model

In [23]:
import importlib, src.train
src.train = importlib.reload(src.train)
from src.train import train

ticker_arg = None if TRAIN_TICKER == 'ALL' else TRAIN_TICKER
label = 'all tickers' if ticker_arg is None else ticker_arg
print(f"Training {cfg['model']['type'].upper()} on {label} ...")

train(cfg, ticker=ticker_arg)
print('\nTraining complete! Checkpoint saved to checkpoints/best_model.pt')


Training LSTM on all tickers ...


batch_loss,▁▁▂▂▁▇▂█▁▁▃▁▁▄▂▁▂▂▄▁▂█▂▂▁▁▂▁▃▁▃▁▁▂▃▁▂▁▁▃
epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
train_loss,█▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▁▂▁▁▁▁▁
val_loss,█▅▄▃▂▁▂▁▁▁▂▂▃▂▂▂▂▂▂▁▂▂▁
batch_loss,0.01159
epoch,23
train_loss,0.01645
val_loss,0.06854


batch_loss,█▅▁▂▃▁▃▁▁▂▂▁▂▁▂▁▂▁▂▁▂▂▁▁▁▁▁▂▁▁▁▁▁▁▂▁▁▁▁▁
epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
test_mae_downside,▁
test_mae_return,▁
test_mae_volatility,▁
test_mse_downside,▁
test_mse_return,▁
test_mse_volatility,▁
test_r2_downside,▁
test_r2_return,▁
+3,...



Training complete! Checkpoint saved to checkpoints/best_model.pt


## Cell 12 — Evaluate on test set

In [24]:
import importlib, src.evaluate
src.evaluate = importlib.reload(src.evaluate)
from src.evaluate import evaluate

eval_ticker = TRAIN_TICKER if TRAIN_TICKER != 'ALL' else cfg['data']['tickers'][0]

metrics, y_pred, y_true = evaluate(cfg, ticker=eval_ticker)

print(f'\nTest Set Metrics - {eval_ticker}')
print('-' * 45)
for k, v in sorted(metrics.items()):
    print(f'  {k:32s}: {v:.4f}')


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


eval/directional_accuracy,▁
eval/downside_mae,▁
eval/downside_r2,▁
eval/downside_rmse,▁
eval/return_ic,▁
eval/return_mae,▁
eval/return_r2,▁
eval/return_rmse,▁
eval/volatility_ic,▁
eval/volatility_mae,▁
+3,...



Test Set Metrics - AAPL
---------------------------------------------
  directional_accuracy            : 0.8742
  downside_ic                     : nan
  downside_mae                    : 0.0427
  downside_r2                     : 0.0000
  downside_rmse                   : 0.1799
  return_ic                       : 0.9377
  return_mae                      : 0.0187
  return_r2                       : 0.8257
  return_rmse                     : 0.0243
  volatility_ic                   : 0.4144
  volatility_mae                  : 0.0039
  volatility_r2                   : -3.0032
  volatility_rmse                 : 0.0064


## Cell 13 — Visualise predictions vs actuals

In [25]:
import plotly.graph_objects as go

n = min(100, len(y_true))
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_true[:n, 0], name='True Return', line=dict(color='#8b949e')))
fig.add_trace(go.Scatter(y=y_pred[:n, 0], name='Pred Return', line=dict(color='#00d4aa')))
fig.update_layout(
    title=f'{eval_ticker} - Return Prediction (first {n} test samples)',
    template='plotly_dark', height=380,
    xaxis_title='Sample', yaxis_title='Log Return',
)
fig.show()


## Cell 14 — Live inference

In [26]:
import importlib, src.inference
src.inference = importlib.reload(src.inference)
from src.inference import run_inference

result = run_inference(eval_ticker, cfg)

print(f"Ticker           : {result['ticker']}")
print(f"Current Price    : ${result.get('current_price', 0):.2f}")
print(f"Predicted Return : {result.get('pred_return', 0)*100:+.2f}%")
print(f"Predicted Vol    : {result.get('pred_volatility', 0)*100:.2f}%")
print(f"Downside Prob    : {result.get('pred_downside_prob', 0)*100:.1f}%")
print(f"Risk Score       : {result.get('risk_score', 0):.0f} / 100  [{result.get('risk_label')}]")
print(f"Outlook          : {result.get('outlook')}")
sf = result.get('sentiment_features', {})
print(f"Sentiment        : {sf.get('weighted_sentiment', 0):+.3f}  ({sf.get('news_volume', 0)} articles)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

ERROR:src.inference:[AAPL] Could not fetch market data.


Ticker           : AAPL
Current Price    : $0.00
Predicted Return : +0.00%
Predicted Vol    : 0.00%
Downside Prob    : 0.0%
Risk Score       : 0 / 100  [None]
Outlook          : None
Sentiment        : +0.000  (0 articles)


## Cell 15 — Backtest

In [27]:
from src.backtest import run_backtest

bt_result = run_backtest(cfg)
m = bt_result['metrics']

print('Backtest Results:')
print(f"  CAGR          : {m.get('cagr', 0)*100:.2f}%")
print(f"  Sharpe Ratio  : {m.get('sharpe_ratio', 0):.3f}")
print(f"  Sortino Ratio : {m.get('sortino_ratio', 0):.3f}")
print(f"  Max Drawdown  : {m.get('max_drawdown', 0)*100:.2f}%")
print(f"  Win Rate      : {m.get('win_rate', 0)*100:.1f}%")
if 'benchmark_cagr' in m:
    print(f"  Benchmark CAGR: {m['benchmark_cagr']*100:.2f}%")


Backtest Results:
  CAGR          : 20.07%
  Sharpe Ratio  : 0.891
  Sortino Ratio : 1.413
  Max Drawdown  : -20.83%
  Win Rate      : 53.8%
  Benchmark CAGR: 14.07%


## Cell 16 — Plot equity curve

In [28]:
from dashboard.dashboard_utils import backtest_chart

ec = bt_result.get('equity_curve')
bm = bt_result.get('benchmark_curve')

if ec is not None:
    fig = backtest_chart(ec, bm)
    if fig:
        fig.show()
else:
    print('No equity curve available.')


## Cell 17 — Save checkpoints to Drive

Run this before your Colab session ends to save your model weights.

In [34]:
import shutil, os

PROJECT_DIR = '/content/drive/MyDrive/ai-portfolio-analyzer'
DRIVE_CKPT = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

saved = []
for fname in os.listdir('checkpoints'):
    src_path = os.path.join('checkpoints', fname)
    dst_path = os.path.join(DRIVE_CKPT, fname)
    shutil.copy2(src_path, dst_path)
    saved.append(fname)

print(f'Saved {len(saved)} checkpoint file(s) to Drive:')
for f in saved:
    print(f'  -> {f}')

import shutil, os

DRIVE_CKPT = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

saved = []
for fname in os.listdir('checkpoints'):
    src_path = os.path.join('checkpoints', fname)
    dst_path = os.path.join(DRIVE_CKPT, fname)
    shutil.copy2(src_path, dst_path)
    saved.append(fname)

print(f'Saved {len(saved)} checkpoint file(s) to Drive:')
for f in saved:
    print(f'  -> {f}')


Saved 3 checkpoint file(s) to Drive:
  -> .gitkeep
  -> best_model.pt
  -> scaler.pkl
Saved 3 checkpoint file(s) to Drive:
  -> .gitkeep
  -> best_model.pt
  -> scaler.pkl
